# Homework: Wikipedia Agent

Build an agent that can search Wikipedia and fetch page content to answer questions.

## Setup: imports

In [ ]:
import requests
from typing import Any, Dict, List

## Setup: Wikipedia tools

A `WikipediaTools` class with `search` and `get_page`. It isn't part of a specific question yet; it's here for when we build the agent.

In [ ]:
class WikipediaTools:
    """
    Provides search and page retrieval utilities over the Wikipedia API.
    """

    headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search Wikipedia for pages related to a query.

        Args:
            query (str): The search query.

        Returns:
            List[Dict[str, Any]]: Search results, each with a page title and snippet.
        """
        response = requests.get(
            "https://en.wikipedia.org/w/api.php",
            params={
                "action": "query",
                "format": "json",
                "list": "search",
                "srsearch": query,
            },
            headers=self.headers,
        )
        response.raise_for_status()
        return response.json()["query"]["search"]

    def get_page(self, title: str) -> str:
        """
        Fetch the raw wikitext content of a Wikipedia page.

        Args:
            title (str): The exact title of the Wikipedia page.

        Returns:
            str: The raw content of the page.
        """
        response = requests.get(
            "https://en.wikipedia.org/w/index.php",
            params={"title": title, "action": "raw"},
            headers=self.headers,
        )
        response.raise_for_status()
        return response.text

In [ ]:
wiki_tools = WikipediaTools()

results = wiki_tools.search("lesser capybara")
[r["title"] for r in results]

In [ ]:
page = wiki_tools.get_page("Capybara")
len(page), page[:500]

## Question 1. Define the Search Tool

Create a `search_wikipedia` function that takes a query, calls the Wikipedia Search API, and returns the results. Test it with "capybara".

**How many total results did the search return?** Count the entries in the `query.search` array.

**Answer: 10**

In [ ]:
headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}


def search_wikipedia(query: str) -> Dict[str, Any]:
    """
    Search Wikipedia for pages related to a query.

    Args:
        query (str): The search query.

    Returns:
        Dict[str, Any]: The search response, with "searchinfo" (total hits)
        and "search" (the list of matching pages).
    """
    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "format": "json",
            "list": "search",
            "srsearch": query,
        },
        headers=headers,
    )
    response.raise_for_status()
    return response.json()["query"]

In [ ]:
results = search_wikipedia("capybara")

# Q1 answer: number of entries in query.search
len(results["search"])

## Question 2. Analyzing Search Results

**How many of the results contain the word "capybara" (case-insensitive) in their title?**

**Answer: 6**

In [ ]:
titles = [r["title"] for r in results["search"]]
matching = [t for t in titles if "capybara" in t.lower()]

# Q2 answer
matching, len(matching)

## Question 3. Define the Get Page Tool

Create a `get_page` function that takes a page title, calls the Wikipedia Get Page API, and returns the page content as a string. Test it with "Capybara".

**How many characters are in the page content?**

**Answer: ~45,000**

In [ ]:
def get_page(title: str) -> str:
    """
    Fetch the raw wikitext content of a Wikipedia page.

    Args:
        title (str): The exact title of the Wikipedia page.

    Returns:
        str: The raw content of the page.
    """
    response = requests.get(
        "https://en.wikipedia.org/w/index.php",
        params={"title": title, "action": "raw"},
        headers=headers,
    )
    response.raise_for_status()
    return response.text

In [ ]:
content = get_page("Capybara")

# Q3 answer
len(content)

## Question 4. Agent Setup

Create an agent with access to both `search_wikipedia` and `get_page`.

**Answer:**
- Framework: PydanticAI
- LLM provider: OpenAI (`gpt-4o-mini`)

In [ ]:
from pydantic_ai import Agent

wiki_instructions = """
You're a Wikipedia research assistant.

Answer the user's question using Wikipedia:
1) Use search_wikipedia to find pages relevant to the question.
2) Use get_page to read the most relevant pages. Fetch more than one if needed.
3) Answer based only on the page content, and cite the page titles you used.
"""

wiki_agent = Agent(
    name="wikipedia",
    model="openai:gpt-4o-mini",
    instructions=wiki_instructions,
    tools=[search_wikipedia, get_page],
)

In [ ]:
# Q4 test run
result = await wiki_agent.run("Where do capybaras live?")
print(result.output)

## Question 5. Testing Your Agent - Single Page

Ask the agent: "What is this page about? https://en.wikipedia.org/wiki/Capybara"

**What is the main topic of the Capybara page according to your agent?**

**Answer:** The page is about the capybara (*Hydrochoerus hydrochaeris*), the largest living rodent. It's a semi-aquatic herbivore native to South America. The page covers its physical traits, social behavior, diet, reproduction, conservation status and cultural significance.

In [ ]:
from pydantic_ai.messages import ToolCallPart

q5_result = await wiki_agent.run("What is this page about? https://en.wikipedia.org/wiki/Capybara")

# Check which tools the agent called
for message in q5_result.all_messages():
    for part in message.parts:
        if isinstance(part, ToolCallPart):
            print("Tool call:", part.tool_name, part.args)

# Q5 answer
print()
print(q5_result.output)

## Question 6. Testing Your Agent - Search Then Fetch

Ask the agent: "What are the main threats to capybara populations?"

**How many total tool calls (search + fetch combined) did your agent make?**

**Answer: 4 tool calls**

Order:
1. `search_wikipedia("Capybara threats population")`
2. `search_wikipedia("Capybara conservation threats")`
3. `get_page("Pantanal")`
4. `get_page("Capybara")`

Final answer (summary): the main threats are hunting and poaching for meat and hides, habitat loss from agriculture, ranching and urban development, and conflict with farmers who see capybaras as competing with livestock. The agent also listed predation, plus climate change, pollution and invasive species (the last three from the Pantanal page).

In [ ]:
q6_result = await wiki_agent.run("What are the main threats to capybara populations?")

# Tool calls in order
tool_calls = [
    part
    for message in q6_result.all_messages()
    for part in message.parts
    if isinstance(part, ToolCallPart)
]
for i, call in enumerate(tool_calls, 1):
    print(f"{i}. {call.tool_name} {call.args}")

# Q6 answer
print()
print("Total tool calls:", len(tool_calls))
print()
print(q6_result.output)